# 1111 職缺匯入 Qdrant

從 `output/` 讀取 enriched JSON，產生 embedding vector，匯入 Qdrant，並進行語意搜尋。

## 設定

In [ ]:
import json
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

from fastembed import TextEmbedding
from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance,
    FieldCondition,
    Filter,
    MatchValue,
    PayloadSchemaType,
    PointStruct,
    VectorParams,
)

INPUT_PATH = Path("output/1111_jobs_page1_top100.enriched.json")
QDRANT_URL = "http://localhost:6333"
COLLECTION_NAME = "jobs_1111_page1_top100"
EMBEDDING_MODEL = "intfloat/multilingual-e5-large"

## 輔助函式

In [ ]:
def join_values(values):
    if not values:
        return ""
    cleaned = [str(v).strip() for v in values if str(v).strip()]
    return "、".join(cleaned)


def parse_salary_range(salary_text):
    if not salary_text:
        return None, None
    numbers = [int(t.replace(",", "")) for t in re.findall(r"[\d,]+", salary_text)]
    if not numbers:
        return None, None
    if len(numbers) == 1:
        return numbers[0], numbers[0]
    return min(numbers), max(numbers)


def build_embedding_text(job):
    require = job.get("require", {})
    job_page = job.get("jobPage", {})
    enum_meaning = job.get("enumMeaning", {})
    sections = [
        ("職缺名稱", job.get("title")),
        ("公司名稱", job.get("companyName")),
        ("產業", job.get("industry", {}).get("name")),
        ("工作地點", job.get("workCity", {}).get("name") or job_page.get("work_location_text")),
        ("職務類別", join_values(enum_meaning.get("roleLabels"))),
        ("工作性質", join_values(enum_meaning.get("jobTypeLabels"))),
        ("工作時間", job_page.get("work_time_text")),
        ("休假制度", job_page.get("vacation_text")),
        ("待遇", job.get("salary")),
        ("學歷要求", require.get("gradesDecoded", {}).get("text")),
        ("工作經驗", require.get("experienceText")),
        ("科系要求", require.get("majorsDecoded", {}).get("text")),
        ("駕照要求", join_values(require.get("drivingLicenseDecoded", {}).get("text_labels"))),
        ("工作技能", job_page.get("skills_text")),
        ("電腦專長", join_values(job_page.get("computer_skill_labels"))),
        ("歡迎身份", join_values(job_page.get("welcome_identity_labels"))),
        ("工作內容", job.get("description")),
    ]
    return "\n".join(f"{label}：{value}" for label, value in sections if value)


def build_payload(job):
    require = job.get("require", {})
    job_page = job.get("jobPage", {})
    enum_meaning = job.get("enumMeaning", {})
    salary_min, salary_max = parse_salary_range(job.get("salary", ""))
    return {
        "job_id": int(job["jobId"]),
        "company_id": int(job["companyId"]),
        "company_name": job.get("companyName"),
        "title": job.get("title"),
        "description": job.get("description"),
        "industry": job.get("industry", {}).get("name"),
        "work_city": job.get("workCity", {}).get("name"),
        "salary_text": job.get("salary"),
        "salary_min": salary_min,
        "salary_max": salary_max,
        "job_url": job.get("jobUrl"),
        "role_labels": enum_meaning.get("roleLabels", []),
        "job_type_labels": enum_meaning.get("jobTypeLabels", []),
        "experience_text": require.get("experienceText"),
        "education_labels": require.get("gradesDecoded", {}).get("labels", []),
        "education_text": require.get("gradesDecoded", {}).get("text"),
        "major_text": require.get("majorsDecoded", {}).get("text"),
        "driving_license_labels": require.get("drivingLicenseDecoded", {}).get("text_labels", []),
        "work_time_text": job_page.get("work_time_text"),
        "vacation_text": job_page.get("vacation_text"),
        "recruit_count": job.get("recruitCount"),
        "recruit_count_text": job.get("recruitCountString"),
        "welcome_identity_labels": job_page.get("welcome_identity_labels", []),
        "computer_skill_labels": job_page.get("computer_skill_labels", []),
        "document_text": build_embedding_text(job),
    }

## 載入資料

In [ ]:
@dataclass(frozen=True)
class PreparedJob:
    id: int
    text: str
    payload: dict


payload = json.loads(INPUT_PATH.read_text(encoding="utf-8"))
prepared_jobs = [
    PreparedJob(id=int(job["jobId"]), text=build_embedding_text(job), payload=build_payload(job))
    for job in payload["jobs"]
]

print(f"載入 {len(prepared_jobs)} 筆職缺")
print("\n--- 第一筆 embedding text ---")
print(prepared_jobs[0].text)

## 產生 Embedding Vector

In [ ]:
model = TextEmbedding(model_name=EMBEDDING_MODEL)
vectors = list(model.embed([job.text for job in prepared_jobs]))

print(f"vector 維度: {len(vectors[0])}")
print(f"共產生 {len(vectors)} 筆 vector")

## 建立 Collection 並匯入 Qdrant

In [ ]:
client = QdrantClient(url=QDRANT_URL)
vector_size = len(vectors[0])

existing = {item.name for item in client.get_collections().collections}
if COLLECTION_NAME not in existing:
    client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(size=vector_size, distance=Distance.COSINE),
    )
    print(f"建立 collection: {COLLECTION_NAME}")
else:
    print(f"collection 已存在: {COLLECTION_NAME}")

for field_name, schema in [
    ("job_id",          PayloadSchemaType.INTEGER),
    ("company_name",    PayloadSchemaType.KEYWORD),
    ("industry",        PayloadSchemaType.KEYWORD),
    ("work_city",       PayloadSchemaType.KEYWORD),
    ("role_labels",     PayloadSchemaType.KEYWORD),
    ("job_type_labels", PayloadSchemaType.KEYWORD),
    ("salary_min",      PayloadSchemaType.INTEGER),
    ("salary_max",      PayloadSchemaType.INTEGER),
]:
    client.create_payload_index(
        collection_name=COLLECTION_NAME,
        field_name=field_name,
        field_schema=schema,
    )

points = [
    PointStruct(id=job.id, vector=vector.tolist(), payload=job.payload)
    for job, vector in zip(prepared_jobs, vectors, strict=True)
]
client.upsert(collection_name=COLLECTION_NAME, points=points, wait=True)
print(f"匯入完成，共 {len(points)} 筆")

## 驗證匯入

In [ ]:
count_result = client.count(collection_name=COLLECTION_NAME, exact=True)
sample_points = client.retrieve(
    collection_name=COLLECTION_NAME,
    ids=[prepared_jobs[0].id],
    with_payload=True,
    with_vectors=False,
)
sample = sample_points[0].payload if sample_points else {}

print(f"預期筆數: {len(prepared_jobs)}")
print(f"實際筆數: {count_result.count}")
print(f"抽樣職缺: {sample.get('title')} / {sample.get('work_city')}")

## 語意搜尋

In [ ]:
QUERY = "會計 全職 台北"
CITY_FILTER = None   # 設城市過濾，例如 "台北市" 或 None
LIMIT = 5

query_vector = list(model.query_embed(QUERY))[0].tolist()

query_filter = None
if CITY_FILTER:
    query_filter = Filter(
        must=[FieldCondition(key="work_city", match=MatchValue(value=CITY_FILTER))]
    )

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    query_filter=query_filter,
    with_payload=True,
    with_vectors=False,
    limit=LIMIT,
)

for point in results.points:
    print(f"[{point.score:.4f}] {point.payload.get('title')} | {point.payload.get('company_name')} | {point.payload.get('work_city')}")